In [ ]:
from lsst.rsp import RSPDiscovery
from astropy.table import Table
discovery = RSPDiscovery("prompt")
service = discovery.get_tap_client()
assert service is not None

In [ ]:
query = """
SELECT TOP 10000 diaObjectId, ra, dec
FROM ppdb.DiaObject
"""
results = service.run_async(query)
assert len(results) > 0

In [ ]:
RADIUS = 0.001
OFFSET = 1.0 / 3600

ut1 = results.to_table()
ut1["dec"] = ut1["dec"] + OFFSET
ut1

In [ ]:
query = f"""
SELECT u.diaObjectId AS upload_id, o.diaObjectId, o.ra, o.dec
FROM ppdb.DiaObject AS o, TAP_UPLOAD.ut1 AS u
WHERE CONTAINS(POINT('ICRS', o.ra, o.dec),
               CIRCLE('ICRS', u.ra, u.dec, {RADIUS})) = 1
"""
job = service.submit_job(query, uploads={"ut1": ut1})
job.run()
job.wait(phases=['COMPLETED', 'ERROR'], timeout=300)
print('Job phase is', job.phase)
print('Job ID is', job.job.jobid)
if job.phase == 'ERROR':
    job.raise_if_error()
results = job.fetch_result().to_table()
own = results[results['upload_id'] == results['diaObjectId']]
assert set(own['upload_id']) == set(ut1['diaObjectId']), own